In [20]:
from Bio.PDB import PDBList 

pdb1 = PDBList()
pdb1.retrieve_pdb_file('1M17', file_format = 'pdb', pdir = '../data/structures')

Structure exists: '../data/structures/pdb1m17.ent' 


'../data/structures/pdb1m17.ent'

In [ ]:
from Bio.PDB import PDBParser

parser = PDBParser(QUIET= True)
erlotinib_structure = parser.get_structure(
    'EGFR_1M17',
    '../data/structures/pdb1m17.ent'
)

for model in structure:
    for chain in model:
        print(chain.id, len(list(chain.get_residues())))


KeyError: 'chain_id'

In [ ]:
for residue in structure[0]['A']:
    if residue.id[0] != ' ':
        print(residue.id, residue.resname)


('H_AQ4', 999, ' ') AQ4
('W', 1, ' ') HOH
('W', 2, ' ') HOH
('W', 3, ' ') HOH
('W', 4, ' ') HOH
('W', 5, ' ') HOH
('W', 6, ' ') HOH
('W', 7, ' ') HOH
('W', 8, ' ') HOH
('W', 9, ' ') HOH
('W', 10, ' ') HOH
('W', 11, ' ') HOH
('W', 12, ' ') HOH
('W', 13, ' ') HOH
('W', 14, ' ') HOH
('W', 15, ' ') HOH
('W', 16, ' ') HOH
('W', 17, ' ') HOH
('W', 18, ' ') HOH
('W', 19, ' ') HOH
('W', 20, ' ') HOH


In [ ]:
#lets find the binding pocket
from Bio.PDB import NeighborSearch
#grab the erlotinib in the structure
ligand = structure[0]['A'][('H_AQ4', 999, ' ')]
atoms = list(structure.get_atoms())
#creating a neighborsearch tool for every atom
ns = NeighborSearch(atoms)
#lets make a list that ignores duplicates, as many ligand atoms will be near the same residue and we only want each pocket, residue counted once
pocket_residues = set()
#lets loop through every atom in the ligandd molecule
for atom in ligand:
    nearby_atoms = ns.search(atom.coord, 5.0)
    #now looping through each of those nearby atoms individually
    for nearby_atom in nearby_atoms:
        parent_residue = nearby_atom.get_parent()

        if parent_residue.id[0]== ' ':
            pocket_residues.add((parent_residue.id[1], parent_residue.resname))

for res in sorted(pocket_residues):
    print(res)


(694, 'LEU')
(695, 'GLY')
(702, 'VAL')
(719, 'ALA')
(721, 'LYS')
(738, 'GLU')
(742, 'MET')
(764, 'LEU')
(765, 'ILE')
(766, 'THR')
(767, 'GLN')
(768, 'LEU')
(769, 'MET')
(770, 'PRO')
(771, 'PHE')
(772, 'GLY')
(773, 'CYS')
(820, 'LEU')
(830, 'THR')
(831, 'ASP')


In [ ]:
import py3Dmol

view = py3Dmol.view(query='pdb:1M17')
view.setStyle({'cartoon': {'color': 'spectrum'}})
view.addStyle({'hetflag': True}, {'stick': {}})

pocket_resi = [res[0] for res in pocket_residues]
view.addStyle({'resi': pocket_resi}, {'stick': {'color': 'orange'}})

view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
#now we turn pocket-binding into a reusable function
from Bio.PDB import PDBList, PDBParser, NeighborSearch
def find_pocket(pdb_id, ligand_code, distance=5.0):
    #download the structure
    pdb1 = PDBList()
    pdb1.retrieve_pdb_file(pdb_id, file_format='pdb', pdir= '../data/structures')
    #parse it
    parser = PDBParser(QUIET=True)
    filepath = f'../data/structures/pdb{pdb_id.lower()}.ent'
    structure = parser.get_structure(pdb_id, filepath)

    #find the ligand
    chain = structure[0]['A']
    ligand = None
    for residue in chain:
        if residue.resname ==ligand_code:
            ligand = residue
            break
    if ligand is None:
        print(f"ligand {ligand_code} is not found in {pdb_id}")
        return None
    #find residues near the ligand
    atoms = list(structure.get_atoms())
    ns = NeighborSearch(atoms)
    pocket_residues = set()
    for atom in ligand:
        nearby_atoms = ns.search(atom.coord, distance)
        for nearby_atom in nearby_atoms:
            parent_residue = nearby_atom.get_parent()
            if parent_residue.id[0] == ' ':
                pocket_residues.add((parent_residue.id[1], parent_residue.resname))
    return pocket_residues

In [ ]:
erlotinib_pocket = find_pocket('1m17','AQ4')
gefitinib_pocket = find_pocket('2ITY', 'IRE')

erlotinib_resnums = set(res[0] for res in erlotinib_pocket)
gefitinib_resnums = set(res[0] for res in gefitinib_pocket)

shared = erlotinib_resnums & gefitinib_resnums
only_erlotinib = erlotinib_resnums - gefitinib_resnums
only_gefitinib = gefitinib_resnums - erlotinib_resnums
print("shared pocket residues:", sorted(shared))
print("residues only in erlotinib pocket:", sorted(only_erlotinib))
print("residues only in gefitinib pocket:", sorted(only_gefitinib))



Structure exists: '../data/structures/pdb1m17.ent' 
Structure exists: '../data/structures/pdb2ity.ent' 
shared pocket residues: [719, 766]
residues only in erlotinib pocket: [694, 695, 702, 721, 738, 742, 764, 765, 767, 768, 769, 770, 771, 772, 773, 820, 830, 831]
residues only in gefitinib pocket: [718, 720, 726, 743, 744, 745, 762, 788, 789, 790, 791, 792, 793, 794, 795, 796, 797, 800, 844, 854, 855]


In [ ]:
#LETS CREATE A CLEAN LIST OF AMINO ACIDS IN EACH OF THE 2 STRUCTURES USING A FUNCTION 
def get_protein_residues(structure, chain_id = 'A'):
    chain = structure[0]['chain_id']
    return [residue for residue in chain if chain_id == ' ']



In [ ]:
#NOW WE TURN THE TWO ORDERED RESIDUE LISTS INTO AMIO-ACID SEQUENCES TO LET US LINE UP THE 2 EGFR STRUCTURES BY THE BIOLOGICAL ORDER RATHER THAN RAW PDB NUMBERS
gefitinib_structure = parser.get_structure('EGFR_2ITY',
    '../data/structures/pdb2ity.ent')

erlotinib_residues = get_protein_residues(
    erlotinib_structure,
    chain_id='A'
)

gefitinib_residues = get_protein_residues(
    gefitinib_structure,
    chain_id='A'
)





KeyError: 'chain_id'

In [ ]:
from Bio.SeqUtils import seq1
def residues_to_sequence(residues):
    return ''.join(seq1(residue.resname) for residue in residues)
erlotinib_sequence = residues_to_sequence(erlotinib)

